# Análisis del Efecto del Aumento del Parque Vehicular Liviano y el Deterioro de la Calidad del Aire con PM2.5 en la Ciudad Capital, Guatemala (2022–2025)

**Metodología:** CRISP-DM (Cross-Industry Standard Process for Data Mining)
**Enfoque:** Series de tiempo univariadas con regresores exógenos (ARIMAX / SARIMAX)
**Variable objetivo (endógena):** Concentración mensual de PM2.5 (μg/m³)

---

Este notebook está organizado estrictamente en las **6 fases de CRISP-DM**:

1. *Business Understanding* — objetivo, hipótesis y métricas de éxito.
2. *Data Understanding* — carga, EDA y visualización cruda.
3. *Data Preparation* — alineación temporal mensual, consolidación y estacionariedad.
4. *Modeling* — ARIMAX y SARIMAX.
5. *Evaluation* — AIC/BIC, RMSE/MAE y diagnóstico de residuales.
6. *Deployment* — interpretación técnica de coeficientes y conclusión estadística.


## Configuración del entorno

Importamos las librerías y fijamos parámetros de reproducibilidad y estilo. Si `pmdarima` no está instalado, las celdas que lo usan tienen un *fallback* manual basado en ACF/PACF y búsqueda en malla (grid search).

```bash
pip install pandas numpy matplotlib seaborn statsmodels scikit-learn pmdarima
```


In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# Configuración global del entorno de análisis
# =============================================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

#from sklearn.metrics import mean_squared_error, mean_absolute_error
# Métricas sin sklearn (evita el bloqueo de Application Control)
def mean_squared_error(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return np.mean((y_true - y_pred) ** 2)

def mean_absolute_error(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return np.mean(np.abs(y_true - y_pred))

# pmdarima es opcional: se usa para auto_arima si está disponible.
try:
    import pmdarima as pm
    HAS_PMDARIMA = True
except Exception:
    HAS_PMDARIMA = False
    print("pmdarima no disponible: se usará grid search / ACF-PACF manual.")

# Reproducibilidad y estética
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 110

# Rutas de los datasets (ajustar según ubicación local)
PATH_VEHICULAR = "./Data/Vehiculos/PARQUE-VEHICULAR-2007-2025-FULL.csv"
PATH_PM25      = "./Data/PM25/PM25_DAIRY_FULL_2022_2025.csv"
PATH_CLIMA     = "./Data/Test/CLIMA_2022_2025.csv"

# Rango temporal de estudio (intersección de los tres datasets)
FECHA_INICIO = "2022-03-01"   # PM2.5 inicia en marzo 2022
FECHA_FIN    = "2025-12-31"


---
# Fase 1 · Business Understanding

## 1.1 Objetivo del proyecto
Determinar y cuantificar la **relación estadística** entre el crecimiento del parque vehicular liviano y la concentración de material particulado fino (PM2.5) en la Ciudad de Guatemala durante el periodo marzo 2022 – diciembre 2025, **controlando por variables climatológicas** (temperatura, precipitación, velocidad del viento y humedad relativa) registradas por la estación central de INSIVUMEH.

## 1.2 Hipótesis de investigación
- **H₀:** El crecimiento del parque vehicular liviano **no** tiene un efecto estadísticamente significativo sobre los niveles de PM2.5 (coeficiente exógeno ≈ 0, p ≥ 0.05).
- **H₁:** El crecimiento del parque vehicular liviano **sí** tiene un efecto positivo y significativo sobre PM2.5 (coeficiente > 0, p < 0.05).

## 1.3 Enfoque estadístico
Se utiliza un marco de **series de tiempo con regresores exógenos**:
- **ARIMAX** — captura la dinámica autorregresiva y de media móvil de PM2.5 más el efecto aislado del parque vehicular.
- **SARIMAX** — añade un componente **estacional anual (s=12)** y el conjunto completo de covariables climatológicas, dado que la calidad del aire en Guatemala exhibe estacionalidad marcada (época seca nov–abr vs. lluviosa may–oct).

## 1.4 Métricas de éxito
| Métrica | Uso |
|---|---|
| **AIC / BIC** | Selección de modelo (penalizan complejidad). Menor es mejor. |
| **RMSE / MAE** | Calidad del ajuste/predicción en μg/m³ sobre el hold-out. |
| **p-valor del coeficiente exógeno** | Criterio central para la hipótesis. |
| **Ljung-Box, Jarque-Bera, Breusch-Pagan** | Validez de los supuestos sobre los residuales. |

## 1.5 Limitaciones reconocidas
- Con ~46 observaciones mensuales el tamaño muestral es modesto para series de tiempo; se requieren **modelos parsimoniosos** y cautela interpretativa.
- El parque vehicular acumulado (`VEH_ACUMULADO`) es una serie **monótonamente creciente** con escasa variación relativa intra-periodo, lo que limita su poder discriminante como regresor.
- Los datos provienen de una sola estación meteorológica (INSIVUMEH central) y un solo monitor de PM2.5, lo que restringe la representatividad espacial.
- Se trata de un estudio **observacional** — cualquier relación encontrada es asociacional, no causal.


---
# Fase 2 · Data Understanding

Se cargan los tres datasets oficiales y se realiza una inspección inicial de estructura, calidad y distribuciones antes de cualquier transformación.

**Fuentes de datos:**
- **Parque vehicular** — SAT Guatemala (registros de alta, 2007–2025), 499,773 registros desagregados por marca, línea, tipo y uso.
- **PM2.5** — Mediciones diarias (μg/m³), marzo 2022 – diciembre 2025, 1,380 observaciones.
- **Clima** — Estación INSIVUMEH central, 2,891 registros diarios con temperatura media, precipitación, velocidad de viento y humedad relativa.


In [ ]:
# =============================================================================
# 2.1 Carga de datasets
# =============================================================================
def cargar_csv(path, **kwargs):
    #\"\"\"Carga robusta de CSV con manejo de encoding latino frecuente en datos GT.\"\"\"
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except (UnicodeDecodeError, LookupError):
            continue
    # Último intento sin especificar encoding
    return pd.read_csv(path, **kwargs)

df_veh   = cargar_csv(PATH_VEHICULAR)
df_pm25  = cargar_csv(PATH_PM25)
df_clima = cargar_csv(PATH_CLIMA)

print("Parque vehicular :", df_veh.shape)
print("PM2.5            :", df_pm25.shape)
print("Clima            :", df_clima.shape)


In [ ]:
# =============================================================================
# 2.2 Inspección de estructura y tipos
# =============================================================================
for nombre, df in [("PARQUE VEHICULAR", df_veh), ("PM2.5", df_pm25), ("CLIMA", df_clima)]:
    print("=" * 70)
    print(nombre)
    print("=" * 70)
    df.info()
    print("\nPrimeras filas:")
    display(df.head())
    print()


In [ ]:
# =============================================================================
# 2.3 Diagnóstico de valores nulos
# =============================================================================
def reporte_nulos(df, nombre):
    nulos = df.isna().sum()
    pct = (nulos / len(df) * 100).round(2)
    rep = pd.DataFrame({"n_nulos": nulos, "pct_nulos": pct})
    rep = rep[rep["n_nulos"] > 0]
    print(f"--- Nulos en {nombre} ---")
    print(rep if not rep.empty else "Sin valores nulos.")
    print()

reporte_nulos(df_veh, "Parque vehicular")
reporte_nulos(df_pm25, "PM2.5")
reporte_nulos(df_clima, "Clima")


In [ ]:
# =============================================================================
# 2.4 Estadística descriptiva de las variables numéricas clave
# =============================================================================
print("Resumen CANTIDAD (parque vehicular):")
display(df_veh["CANTIDAD"].describe())

print("\nResumen PM2.5 diario:")
display(df_pm25["PM25"].describe())

print("\nResumen variables climáticas:")
display(df_clima.describe())


### Hallazgos del EDA preliminar

**Parque vehicular:** Los 499,773 registros contienen 0 nulos. La columna `CANTIDAD` tiene media de 2.86 y mediana de 1, con un máximo de 1,889 — esto confirma que la mayoría de combinaciones marca/modelo/mes registran entre 1 y 2 unidades, con agrupaciones puntuales de modelos populares. La suma mensual (no el conteo de filas) es la variable correcta.

**PM2.5:** La serie diaria presenta media de 30.43 μg/m³ y mediana de 18.15, con una **asimetría positiva pronunciada** (la media es casi el doble de la mediana). El valor máximo de 1,016 μg/m³ es un **outlier extremo** — concentraciones de esa magnitud son atípicas incluso en episodios severos de quema agrícola o incendios forestales, y probablemente corresponde a un evento puntual o error de sensor. La desviación estándar (42.24) es mayor que la media, indicando alta variabilidad diaria que la agregación mensual suavizará.

**Clima:** Se identifican **anomalías críticas en los datos de INSIVUMEH**:
- `PRECIPITACIÓN` tiene un valor mínimo de **-99.9 mm**, que es un código centinela estándar para "dato faltante" en estaciones meteorológicas. Debe tratarse como nulo antes de la agregación. Solo hay 1 nulo declarado de 2,890 registros, pero los -99.9 enmascaran valores faltantes adicionales.
- `HUMEDAD_RELATIVA` presenta un máximo de **108%**, físicamente imposible (el máximo teórico es 100%), sugiriendo un error de calibración o transcripción puntual.
- `VELOCIDAD_VIENTO` oscila entre 0.0 y 60.7 km/h con media de 12.3, un rango plausible para la estación central.
- `TEMPERATURA_MEDIA` varía entre 14.0 y 25.7°C, consistente con el clima subtropical de montaña de la Ciudad de Guatemala (~1,500 msnm).

Estos problemas se corregirán en la Fase 3 (Data Preparation).


In [ ]:
# =============================================================================
# 2.5 Construcción de índices temporales para visualización cruda
#     (sin todavía hacer la consolidación final; solo para graficar)
# =============================================================================

# --- PM2.5: construir fecha diaria a partir de YEAR/MONTH/DAY ---
pm25_raw = df_pm25.copy()
pm25_raw["FECHA"] = pd.to_datetime(
    dict(year=pm25_raw["YEAR"], month=pm25_raw["MONTH"], day=pm25_raw["DAY"]),
    errors="coerce"
)
pm25_raw = pm25_raw.dropna(subset=["FECHA"]).sort_values("FECHA")

# --- Clima: parsear FECHA ---
clima_raw = df_clima.copy()
clima_raw["FECHA"] = pd.to_datetime(clima_raw["FECHA"], errors="coerce", dayfirst=True)
clima_raw = clima_raw.dropna(subset=["FECHA"]).sort_values("FECHA")

# --- Parque vehicular: fecha mensual a partir de ANIO_ALZA/MES ---
veh_raw = df_veh.copy()
veh_mensual_raw = (
    veh_raw.groupby(["ANIO_ALZA", "MES"], as_index=False)["CANTIDAD"].sum()
)
veh_mensual_raw["FECHA"] = pd.to_datetime(
    dict(year=veh_mensual_raw["ANIO_ALZA"], month=veh_mensual_raw["MES"], day=1),
    errors="coerce"
)
veh_mensual_raw = veh_mensual_raw.dropna(subset=["FECHA"]).sort_values("FECHA")

print("Rango PM2.5 :", pm25_raw['FECHA'].min().date(), "→", pm25_raw['FECHA'].max().date())
print("Rango Clima :", clima_raw['FECHA'].min().date(), "→", clima_raw['FECHA'].max().date())
print("Rango Veh.  :", veh_mensual_raw['FECHA'].min().date(), "→", veh_mensual_raw['FECHA'].max().date())


### Cobertura temporal de las fuentes

- **PM2.5:** 4 de marzo 2022 → 31 de diciembre 2025 (≈3 años y 10 meses). Nótese que inicia el 4 de marzo, no el 1, por lo que el mes de marzo 2022 tiene cobertura parcial.
- **Clima (INSIVUMEH):** 1 de enero 2022 → 12 de diciembre 2025. El cierre en diciembre 12 implica que diciembre 2025 tendrá un promedio parcial (12 de 31 días).
- **Parque vehicular:** 1 de enero 1980 → 1 de diciembre 2025. El rango completo excede ampliamente el periodo de estudio; se recortará a marzo 2022 en la Fase 3.

La **intersección temporal útil** es **marzo 2022 – diciembre 2025** (46 meses), dictada por la disponibilidad del PM2.5.


In [ ]:
# =============================================================================
# 2.6 Visualización de las series en estado CRUDO
# =============================================================================
fig, axes = plt.subplots(3, 1, figsize=(13, 11))

axes[0].plot(pm25_raw["FECHA"], pm25_raw["PM25"], color="#c0392b", lw=0.8)
axes[0].set_title("PM2.5 diario (crudo) — μg/m³")
axes[0].set_ylabel("μg/m³")

axes[1].plot(clima_raw["FECHA"], clima_raw["TEMPERATURA_MEDIA"], color="#e67e22", lw=0.8, label="Temp. media (°C)")
axes[1].plot(clima_raw["FECHA"], clima_raw["HUMEDAD_RELATIVA"], color="#2980b9", lw=0.8, alpha=0.7, label="Humedad (%)")
axes[1].set_title("Variables climáticas seleccionadas (crudo)")
axes[1].legend(loc="upper right")

axes[2].bar(veh_mensual_raw["FECHA"], veh_mensual_raw["CANTIDAD"], width=20, color="#27ae60")
axes[2].set_title("Parque vehicular liviano — altas mensuales (suma de CANTIDAD)")
axes[2].set_ylabel("Vehículos")

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# 2.7 Distribuciones de las variables principales
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(pm25_raw["PM25"], kde=True, ax=axes[0], color="#c0392b")
axes[0].set_title("Distribución PM2.5 diario")

sns.histplot(clima_raw["PRECIPITACIÓN"], kde=True, ax=axes[1], color="#2980b9")
axes[1].set_title("Distribución Precipitación diaria")

sns.histplot(veh_mensual_raw["CANTIDAD"], kde=True, ax=axes[2], color="#27ae60")
axes[2].set_title("Distribución altas vehiculares mensuales")
plt.tight_layout()
plt.show()


---
# Fase 3 · Data Preparation

Esta es la fase **más crítica** del análisis. Objetivos:

1. **Limpieza de anomalías** identificadas en la Fase 2: valores centinela de precipitación (-99.9) y humedades >100%.
2. **Alineación temporal** a granularidad **mensual** (`DatetimeIndex`, frecuencia `MS`).
3. **Reglas de agregación diaria → mensual:** PM2.5/temperatura/humedad/viento → promedio mensual; precipitación → suma mensual (lámina acumulada).
4. **Construcción de la variable de stock vehicular** — altas mensuales acumuladas (`VEH_ACUMULADO`).
5. **Pruebas de estacionariedad** (ADF, KPSS) y determinación del orden de diferenciación.


## 3.0 Limpieza de anomalías detectadas en el EDA

Antes de la agregación mensual, corregimos los problemas de calidad de datos identificados.


In [ ]:
# =============================================================================
# 3.0 Limpieza de anomalías en datos crudos
# =============================================================================

# --- Precipitación: reemplazar -99.9 (código centinela) por NaN ---
n_sentinel = (clima_raw["PRECIPITACIÓN"] == -99.9).sum()
clima_raw.loc[clima_raw["PRECIPITACIÓN"] <= 0, "PRECIPITACIÓN"] = np.nan
print(f"Precipitación: {n_sentinel} valores -99.9 convertidos a NaN")

# --- Humedad relativa: valores >100% se capean a 100 ---
n_hum = (clima_raw["HUMEDAD_RELATIVA"] > 100).sum()
clima_raw.loc[clima_raw["HUMEDAD_RELATIVA"] > 100, "HUMEDAD_RELATIVA"] = 100
print(f"Humedad relativa: {n_hum} valores >100% capeados a 100")

# --- PM2.5: outliers extremos (>500 μg/m³ = categoría 'hazardous' de EPA) ---
n_out_pm = (pm25_raw["PM25"] > 500).sum()
pm25_raw.loc[pm25_raw["PM25"] > 500, "PM25"] = np.nan
print(f"PM2.5: {n_out_pm} valores >500 μg/m³ tratados como outliers (→ NaN)")
print(f"  Estos se interpolarán al agregar a nivel mensual.")


In [ ]:
# =============================================================================
# 3.1 Agregación mensual de cada fuente
# =============================================================================

# --- PM2.5: promedio mensual ---
pm25_m = (
    pm25_raw.set_index("FECHA")["PM25"]
    .resample("MS")          # MS = Month Start
    .mean()
    .rename("PM25")
)

# --- Clima: agregaciones diferenciadas por variable ---
clima_idx = clima_raw.set_index("FECHA")
clima_m = pd.DataFrame({
    "TEMPERATURA_MEDIA": clima_idx["TEMPERATURA_MEDIA"].resample("MS").mean(),
    "HUMEDAD_RELATIVA":  clima_idx["HUMEDAD_RELATIVA"].resample("MS").mean(),
    "VELOCIDAD_VIENTO":  clima_idx["VELOCIDAD_VIENTO"].resample("MS").mean(),
    "PRECIPITACION":     clima_idx["PRECIPITACIÓN"].resample("MS").sum(),   # suma mensual
})

# --- Parque vehicular: ya está mensual; lo indexamos a MS ---
veh_m = (
    veh_mensual_raw.set_index("FECHA")["CANTIDAD"]
    .resample("MS").sum()
    .rename("VEH_ALTAS")            # flujo: altas del mes
)

print("PM2.5 mensual:", pm25_m.shape)
print("Clima mensual:", clima_m.shape)
print("Veh.  mensual:", veh_m.shape)


In [ ]:
# =============================================================================
# 3.2 Filtrado temporal al rango común y consolidación
# =============================================================================
# Recortamos el parque vehicular (que cubre 2007-2025) al rango de estudio.
veh_m   = veh_m.loc[FECHA_INICIO:FECHA_FIN]
pm25_m  = pm25_m.loc[FECHA_INICIO:FECHA_FIN]
clima_m = clima_m.loc[FECHA_INICIO:FECHA_FIN]

# Unión por índice mensual (inner join => intersección exacta de fechas)
df = pd.concat([pm25_m, veh_m, clima_m], axis=1, join="inner").sort_index()

# Variable de STOCK (parque acumulado) = altas acumuladas dentro del periodo.
# Si dispones del parque base a inicio de 2022, súmalo aquí como offset.
df["VEH_ACUMULADO"] = df["VEH_ALTAS"].cumsum()

# Aseguramos frecuencia mensual explícita en el índice (clave para statsmodels)
df = df.asfreq("MS")

print("DataFrame consolidado:", df.shape)
display(df.head())
display(df.tail())


In [ ]:
# =============================================================================
# 3.3 Tratamiento de nulos residuales tras la unión
#     (meses sin medición se interpolan temporalmente)
# =============================================================================
print("Nulos por columna tras la unión:")
print(df.isna().sum(), "\n")

# Interpolación temporal lineal + relleno de extremos (parsimonioso dado n pequeño)
df = df.interpolate(method="time").bfill().ffill()

print("Nulos tras interpolación:", df.isna().sum().sum())
print("Número de observaciones mensuales:", len(df))


### Verificación del DataFrame consolidado

El DataFrame resultante tiene **46 observaciones mensuales** (marzo 2022 – diciembre 2025), **0 nulos** en todas las columnas, y las siguientes variables:

- `PM25` — concentración media mensual (μg/m³), rango observado ≈10–52 μg/m³
- `VEH_ALTAS` — vehículos livianos dados de alta en el mes (flujo), rango ≈8,600–18,000
- `VEH_ACUMULADO` — suma acumulada de altas desde marzo 2022 (stock proxy), crece de 9,019 a 470,249
- `TEMPERATURA_MEDIA` — promedio mensual en °C (≈19–22°C)
- `HUMEDAD_RELATIVA` — promedio mensual en % (≈65–80%)
- `VELOCIDAD_VIENTO` — promedio mensual en km/h
- `PRECIPITACION` — lámina acumulada mensual en mm

**Nota importante sobre `VEH_ACUMULADO`:** Esta variable es una suma acumulada *dentro del periodo*, no el parque vehicular total del país. Representa las altas incrementales de vehículos livianos a partir de marzo 2022 y crece **monótonamente**, lo cual tiene implicaciones directas para la estacionariedad y la interpretación del modelo.


In [ ]:
# =============================================================================
# 3.4 Matriz de correlación exploratoria
# =============================================================================
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=.5)
plt.title("Correlación lineal entre variables mensuales")
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# 3.5 Serie consolidada de PM2.5 con su exógena principal
# =============================================================================
fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(df.index, df["PM25"], color="#c0392b", marker="o", ms=3, label="PM2.5 (μg/m³)")
ax1.set_ylabel("PM2.5 (μg/m³)", color="#c0392b")
ax1.tick_params(axis="y", labelcolor="#c0392b")

ax2 = ax1.twinx()
ax2.plot(df.index, df["VEH_ACUMULADO"], color="#27ae60", lw=2, label="Parque acumulado")
ax2.set_ylabel("Parque vehicular acumulado", color="#27ae60")
ax2.tick_params(axis="y", labelcolor="#27ae60")

plt.title("PM2.5 mensual vs. parque vehicular acumulado")
fig.tight_layout()
plt.show()


## 3.6 Pruebas de estacionariedad (ADF y KPSS)

La estacionariedad es requisito para los componentes ARMA del modelo. Aplicamos dos pruebas complementarias:

- **ADF (Dickey-Fuller Aumentada):** H₀ = raíz unitaria (no estacionaria). Si p < 0.05 → rechazamos → la serie es estacionaria.
- **KPSS:** H₀ = la serie **es** estacionaria. Si p < 0.05 → rechazamos → no estacionaria. (Hipótesis invertida respecto a ADF.)

Usar ambas reduce la ambigüedad cuando solo una prueba está en zona limítrofe.


In [ ]:
# =============================================================================
# 3.6 Funciones de prueba de estacionariedad
# =============================================================================
def test_adf(serie, nombre=""):
    #\"\"\"Dickey-Fuller Aumentada. p<0.05 => estacionaria.\"\"\"
    serie = serie.dropna()
    stat, p, lags, nobs, crit, _ = adfuller(serie, autolag="AIC")
    veredicto = "ESTACIONARIA" if p < 0.05 else "NO estacionaria"
    print(f"[ADF] {nombre:18s} | estad={stat:7.3f} | p={p:6.4f} | {veredicto}")
    return p

def test_kpss(serie, nombre=""):
    #\"\"\"KPSS. p<0.05 => NO estacionaria (H0 = estacionaria).\"\"\"
    serie = serie.dropna()
    stat, p, lags, crit = kpss(serie, regression="c", nlags="auto")
    veredicto = "NO estacionaria" if p < 0.05 else "ESTACIONARIA"
    print(f"[KPSS]{nombre:18s} | estad={stat:7.3f} | p={p:6.4f} | {veredicto}")
    return p

print("=== Series en nivel ===")
for col in ["PM25", "VEH_ACUMULADO", "VEH_ALTAS", "TEMPERATURA_MEDIA"]:
    test_adf(df[col], col)
print()
for col in ["PM25", "VEH_ACUMULADO"]:
    test_kpss(df[col], col)


### Interpretación de las pruebas de estacionariedad

| Variable | ADF estad. | ADF p-valor | KPSS estad. | KPSS p-valor | Veredicto |
|---|---|---|---|---|---|
| **PM2.5** | -3.474 | 0.0087 | 0.095 | 0.10 | **Estacionaria** (ambas pruebas coinciden) |
| **VEH_ACUMULADO** | 5.132 | 1.0000 | 1.025 | 0.01 | **No estacionaria** (ambas rechazan) |
| VEH_ALTAS | -0.101 | 0.9494 | — | — | No estacionaria |
| TEMPERATURA_MEDIA | -1.927 | 0.3196 | — | — | No estacionaria |

**Hallazgos clave:**

1. **PM2.5 es estacionaria en nivel** (ADF p=0.0087, KPSS p=0.10). No requiere diferenciación → **d=0**. Esto es coherente con la naturaleza del contaminante: no hay una tendencia secular de largo plazo en 46 meses, sino oscilaciones alrededor de un nivel medio influidas por el ciclo estacional seca/lluvia.

2. **VEH_ACUMULADO es fuertemente no estacionaria** (ADF estad.=+5.132, p=1.0). Esto era inevitable: es una suma acumulativa que solo puede crecer. El estadístico ADF *positivo* confirma un comportamiento de raíz unitaria sin reversa a la media. Esto plantea un **riesgo de regresión espuria** si se usa como regresor de una serie estacionaria (PM2.5). El modelo SARIMAX con D=1 (diferenciación estacional) mitiga parcialmente este problema al operar en primeras diferencias estacionales.

3. **VEH_ALTAS tampoco es estacionaria** (p=0.949), aunque no es una tendencia monotónica sino probablemente una serie con cambio estructural o alta persistencia.


In [ ]:
# =============================================================================
# 3.7 Determinación del orden de diferenciación 'd' para PM2.5
# =============================================================================
def orden_diferenciacion(serie, max_d=2, alpha=0.05):
    #\"\"\"Devuelve el menor d tal que la serie diferenciada es estacionaria por ADF.\"\"\"
    s = serie.dropna().copy()
    for d in range(max_d + 1):
        p = adfuller(s, autolag="AIC")[1]
        if p < alpha:
            return d
        s = s.diff().dropna()
    return max_d

d_pm25 = orden_diferenciacion(df["PM25"])
print(f"Orden de diferenciación sugerido para PM2.5: d = {d_pm25}")

# Visualizar PM2.5 diferenciada (si d>=1)
if d_pm25 >= 1:
    pm25_diff = df["PM25"].diff().dropna()
    test_adf(pm25_diff, "PM2.5 diff(1)")
    plt.figure(figsize=(13, 4))
    plt.plot(pm25_diff.index, pm25_diff.values, color="#8e44ad", marker="o", ms=3)
    plt.axhline(0, color="k", lw=.8)
    plt.title("PM2.5 — primera diferencia")
    plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 3.8 ACF y PACF para guiar la selección de (p, q)
# =============================================================================
serie_acf = df["PM25"].diff(d_pm25).dropna() if d_pm25 >= 1 else df["PM25"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(serie_acf, lags=min(18, len(serie_acf)//2 - 1), ax=axes[0])
axes[0].set_title("ACF (sugiere q / Q)")
plot_pacf(serie_acf, lags=min(18, len(serie_acf)//2 - 1), ax=axes[1], method="ywm")
axes[1].set_title("PACF (sugiere p / P)")
plt.tight_layout(); plt.show()


## 3.9 Separación train / test y matriz exógena

Reservamos los **últimos 6 meses** como conjunto de prueba (*hold-out*) para evaluar capacidad predictiva fuera de muestra. Definimos dos conjuntos de exógenas:

- **Exógena ARIMAX:** sólo el parque vehicular (`VEH_ACUMULADO`).
- **Exógenas SARIMAX:** parque vehicular + clima completo.

Las exógenas se **estandarizan** (z-score con estadísticos de *train*) para que sus coeficientes sean comparables y el optimizador converja mejor.


In [ ]:
# =============================================================================
# 3.9 Split temporal y estandarización de exógenas
# =============================================================================
H_TEST = 6                               # meses de hold-out
y = df["PM25"].astype(float)

EXOG_ARIMAX  = ["VEH_ACUMULADO"]
EXOG_SARIMAX = ["VEH_ACUMULADO", "TEMPERATURA_MEDIA", "PRECIPITACION",
                "VELOCIDAD_VIENTO", "HUMEDAD_RELATIVA"]

y_train, y_test = y.iloc[:-H_TEST], y.iloc[-H_TEST:]

def estandarizar(train_df, test_df):
    #\"\"\"z-score usando media/std del TRAIN para no filtrar información.\"\"\"
    mu, sd = train_df.mean(), train_df.std(ddof=0).replace(0, 1)
    return (train_df - mu) / sd, (test_df - mu) / sd

X1_tr, X1_te = estandarizar(df[EXOG_ARIMAX].iloc[:-H_TEST],  df[EXOG_ARIMAX].iloc[-H_TEST:])
X2_tr, X2_te = estandarizar(df[EXOG_SARIMAX].iloc[:-H_TEST], df[EXOG_SARIMAX].iloc[-H_TEST:])

print(f"Train: {len(y_train)} meses | Test: {len(y_test)} meses")
print("Exógenas ARIMAX :", EXOG_ARIMAX)
print("Exógenas SARIMAX:", EXOG_SARIMAX)


---
# Fase 4 · Modeling

Construimos los dos modelos sobre el conjunto de entrenamiento. En ambos la **endógena es PM2.5**.


## 4.1 Modelo 1 — ARIMAX

`PM2.5 ~ ARIMA(p,d,q)` con un solo regresor exógeno: **parque vehicular acumulado** (`VEH_ACUMULADO`).

Se buscan los órdenes `(p, d, q)` que minimizan el AIC mediante una búsqueda en malla (grid search), dado que `pmdarima` no está disponible en este entorno. El orden de diferenciación `d` se fija en 0, según los resultados de ADF/KPSS de la Fase 3.


In [ ]:
# =============================================================================
# 4.1 Selección de (p,d,q) para ARIMAX
# =============================================================================
def grid_search_arima(y, exog, d, p_range=range(0,4), q_range=range(0,4),
                      seasonal_order=(0,0,0,0)):
    #\"\"\"Búsqueda en malla minimizando AIC. Fallback sin pmdarima.\"\"\"
    mejor = {"aic": np.inf, "order": None, "res": None}
    for p in p_range:
        for q in q_range:
            try:
                res = SARIMAX(y, exog=exog, order=(p, d, q),
                              seasonal_order=seasonal_order,
                              enforce_stationarity=False,
                              enforce_invertibility=False).fit(disp=False)
                if res.aic < mejor["aic"]:
                    mejor = {"aic": res.aic, "order": (p, d, q), "res": res}
            except Exception:
                continue
    return mejor

if HAS_PMDARIMA:
    auto1 = pm.auto_arima(
        y_train, exogenous=X1_tr, d=d_pm25, seasonal=False,
        stepwise=True, suppress_warnings=True, error_action="ignore",
        max_p=4, max_q=4, information_criterion="aic"
    )
    order_arimax = auto1.order
    print("auto_arima → order ARIMAX:", order_arimax)
else:
    best1 = grid_search_arima(y_train, X1_tr, d_pm25)
    order_arimax = best1["order"]
    print("grid search → order ARIMAX:", order_arimax)


In [ ]:
# =============================================================================
# 4.1b Ajuste final del ARIMAX y resumen
# =============================================================================
arimax = SARIMAX(
    y_train, exog=X1_tr, order=order_arimax,
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)

print(arimax.summary())


### Análisis del modelo ARIMAX: SARIMAX(1,0,3)

**Diagnóstico de convergencia — ALERTA CRÍTICA:**

El modelo presenta una **advertencia de matriz de covarianza singular o casi-singular** (número de condición = 7.72×10¹⁴). Esto tiene consecuencias directas:

- Los errores estándar de los coeficientes MA son absurdamente grandes: `ma.L1` tiene std err = 1,896.75, `ma.L2` = 1,473.86, `ma.L3` = 1,308.64. Estos valores invalidan cualquier inferencia basada en los p-valores de esos términos.
- La varianza del error (`sigma2 = 233.03`) también tiene un std err de 442,000, haciendo que su p-valor sea 1.000.
- El coeficiente de `VEH_ACUMULADO` (4.51) tiene un error estándar de 58.88 y un **p-valor de 0.939**, con un intervalo de confianza [−110.89, 119.92] que cruza el cero ampliamente. No hay evidencia alguna de relación en este modelo, pero el resultado es **no interpretable** debido a la inestabilidad numérica.

**¿Por qué ocurre la singularidad?** El modelo ARIMA(1,0,3) con d=0 intenta ajustar 3 coeficientes MA más un AR(1) a una serie de solo 40 observaciones con un único regresor que crece monotónicamente. Los 3 términos MA compiten por explicar la misma estructura de autocorrelación, generando colinealidad perfecta entre parámetros. El AR(1) cercano a la unidad (0.9857) indica que el modelo se apoya casi totalmente en la inercia autorregresiva y los MA son redundantes.

**Métricas de ajuste:** AIC = 313.78, BIC = 323.28, LogLik = −150.89. Estos valores servirán de línea base para comparación, pero las métricas de información no corrigen por la inestabilidad numérica del modelo.


## 4.2 Modelo 2 — SARIMAX

`PM2.5 ~ SARIMA(p,d,q)(P,D,Q,12)` con **parque vehicular + variables climáticas** como regresores exógenos.

La estacionalidad anual `s=12` modela el patrón cíclico seca/lluviosa que domina la variabilidad de PM2.5 en Guatemala. Se buscan los órdenes regular y estacional por grid search acotado, dado el tamaño muestral reducido (40 obs. de entrenamiento, ~3.3 ciclos estacionales).


In [ ]:
# =============================================================================
# 4.2 Selección de órdenes para SARIMAX (regular + estacional, s=12)
# =============================================================================
S = 12

if HAS_PMDARIMA:
    auto2 = pm.auto_arima(
        y_train, exogenous=X2_tr, d=d_pm25, seasonal=True, m=S,
        D=None, stepwise=True, suppress_warnings=True, error_action="ignore",
        max_p=3, max_q=3, max_P=2, max_Q=2, information_criterion="aic"
    )
    order_sarimax     = auto2.order
    seasonal_sarimax  = auto2.seasonal_order
    print("auto_arima → order:", order_sarimax, "| seasonal:", seasonal_sarimax)
else:
    # Grid acotado por el n pequeño: probamos combinaciones estacionales simples.
    mejor = {"aic": np.inf, "order": None, "seasonal": None, "res": None}
    for P in (0, 1):
        for Q in (0, 1):
            for D in (0, 1):
                b = grid_search_arima(y_train, X2_tr, d_pm25,
                                      p_range=range(0,3), q_range=range(0,3),
                                      seasonal_order=(P, D, Q, S))
                if b["res"] is not None and b["aic"] < mejor["aic"]:
                    mejor = {"aic": b["aic"], "order": b["order"],
                             "seasonal": (P, D, Q, S), "res": b["res"]}
    order_sarimax    = mejor["order"]
    seasonal_sarimax = mejor["seasonal"]
    print("grid search → order:", order_sarimax, "| seasonal:", seasonal_sarimax)


In [ ]:
# =============================================================================
# 4.2b Ajuste final del SARIMAX y resumen
# =============================================================================
sarimax = SARIMAX(
    y_train, exog=X2_tr, order=order_sarimax,
    seasonal_order=seasonal_sarimax,
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)

print(sarimax.summary())


### Análisis del modelo SARIMAX: (0,0,2)×(1,1,1,12)

**Convergencia:** A diferencia del ARIMAX, este modelo **no presenta la advertencia de matriz singular**, lo que significa que los errores estándar y p-valores son numéricamente estables y estadísticamente interpretables.

**Especificación seleccionada:** La búsqueda en malla eligió D=1 (una diferenciación estacional), lo que implica que el modelo opera sobre la diferencia estacional de PM2.5: `ΔPM2.5_t = PM2.5_t − PM2.5_{t−12}`. Esto es adecuado para remover el ciclo anual, aunque con solo ~3.3 ciclos completos la estimación estacional es frágil.

**Coeficientes exógenos — hallazgo central del estudio:**

| Variable | Coef. | p-valor | Interpretación |
|---|---|---|---|
| `VEH_ACUMULADO` | **−19.10** | **0.003** *** | Significativo al 1%, pero con **signo negativo** — contrario a la hipótesis H₁ |
| `TEMPERATURA_MEDIA` | **+21.23** | **0.000** *** | Significativo al 0.1%. Mayor temperatura → mayor PM2.5 |
| `PRECIPITACION` | −2.49 | 0.188 | No significativo. Dirección esperada (lluvia lava partículas) |
| `VELOCIDAD_VIENTO` | +4.63 | 0.348 | No significativo |
| `HUMEDAD_RELATIVA` | +17.08 | 0.119 | Marginalmente no significativo (p ≈ 0.12) |

**El resultado inesperado de `VEH_ACUMULADO`:**

El coeficiente **negativo** (−19.10, p=0.003) indica que, tras controlar por clima y estacionalidad, un aumento de una desviación estándar en el parque acumulado se asocia con una **disminución** de 19.1 μg/m³ en PM2.5. Esto es **contrario a la hipótesis de investigación**. Posibles explicaciones:

1. **Confusión con la tendencia temporal.** `VEH_ACUMULADO` crece monotónicamente (de 9,019 a 470,249). Si PM2.5 tiene una leve tendencia descendente residual tras la diferenciación estacional (por ejemplo, por mejoras en combustibles o renovación de la flota), el modelo no puede distinguir "más vehículos" de "paso del tiempo". El coeficiente negativo podría estar capturando un efecto temporal espurio.

2. **Renovación de flota.** Los vehículos más nuevos (que son los que se dan de alta) tienen emisiones per-vehículo significativamente menores que los más antiguos. Un parque que crece con vehículos modernos podría no incrementar las emisiones netas.

3. **Colinealidad con las variables climáticas.** La temperatura es significativa y positiva (+21.23), y correlaciona con la época seca, que es cuando también hay más altas vehiculares. La inclusión conjunta puede redistribuir el efecto estacional entre regresores.

**Métricas de ajuste:** AIC = 89.58, BIC = 95.23, LogLik = −34.79. Sustancialmente mejores que el ARIMAX (AIC 313.78), lo cual refleja que las variables climáticas y la estacionalidad capturan la mayor parte de la variabilidad de PM2.5.

**Diagnósticos del resumen:** Ljung-Box L1 p=0.01 (autocorrelación residual) y Heteroskedasticidad p=0.01 (heterocedasticidad) señalan violaciones de supuestos que deben examinarse en la Fase 5.


---
# Fase 5 · Evaluation

Comparamos ambos modelos en dos dimensiones: **criterios de información** (AIC/BIC, penalizan complejidad sobre el conjunto de entrenamiento) y **error predictivo** (RMSE/MAE sobre el hold-out de 6 meses). Luego diagnosticamos los residuales del modelo seleccionado.


In [ ]:
# =============================================================================
# 5.1 Comparación AIC / BIC
# =============================================================================
comp = pd.DataFrame({
    "Modelo": ["ARIMAX", "SARIMAX"],
    "Orden":  [str(order_arimax), f"{order_sarimax}x{seasonal_sarimax}"],
    "AIC":    [arimax.aic, sarimax.aic],
    "BIC":    [arimax.bic, sarimax.bic],
    "LogLik": [arimax.llf, sarimax.llf],
}).set_index("Modelo")
display(comp.round(2))


### Interpretación AIC / BIC

El SARIMAX domina por **amplio margen**: AIC = 89.58 vs. 313.78, BIC = 95.23 vs. 323.28, una diferencia de >220 puntos en AIC. En escala de Akaike, una diferencia >10 se considera evidencia contundente en favor del modelo con menor AIC. La log-verosimilitud del SARIMAX (−34.79) es dramáticamente superior a la del ARIMAX (−150.89), lo que confirma que la inclusión de variables climáticas y estacionalidad capturan estructura real de la serie, no solo ruido.

Sin embargo, AIC y BIC evalúan ajuste *in-sample* penalizado. La prueba definitiva es el rendimiento predictivo fuera de muestra.


In [ ]:
# =============================================================================
# 5.2 Pronóstico sobre el hold-out y métricas RMSE / MAE
# =============================================================================
def evaluar_forecast(modelo, X_test, y_test, nombre):
    fc = modelo.get_forecast(steps=len(y_test), exog=X_test)
    y_hat = fc.predicted_mean
    rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    mae  = mean_absolute_error(y_test, y_hat)
    print(f"{nombre:9s} | RMSE={rmse:6.3f} | MAE={mae:6.3f}")
    return y_hat, rmse, mae

yhat1, rmse1, mae1 = evaluar_forecast(arimax,  X1_te, y_test, "ARIMAX")
yhat2, rmse2, mae2 = evaluar_forecast(sarimax, X2_te, y_test, "SARIMAX")

# Selección del modelo ganador (criterio combinado: menor RMSE de test, desempate por AIC)
if rmse2 <= rmse1:
    ganador, yhat_g, nombre_g = sarimax, yhat2, "SARIMAX"
else:
    ganador, yhat_g, nombre_g = arimax, yhat1, "ARIMAX"
print("\nModelo ganador:", nombre_g)


### Interpretación del error predictivo

**Paradoja AIC vs. RMSE:** El SARIMAX gana abrumadoramente en AIC (89 vs. 314) pero pierde en RMSE de hold-out (47.7 vs. 15.7 μg/m³). Esto no es contradictorio y revela un fenómeno clásico:

- El SARIMAX tiene 10 parámetros ajustados sobre solo 40 observaciones (ratio parámetros/n = 0.25, muy alto). Con D=1 estacional, el modelo efectivo tiene aún menos grados de libertad. **Sobreajuste in-sample, mal desempeño out-of-sample** — el modelo memorizó patrones del entrenamiento que no se generalizan al hold-out.
- El ARIMAX, con su estructura más simple y su AR(1) cercano a 1, actúa esencialmente como un modelo de "la mejor predicción de PM2.5 del mes siguiente es el valor actual" (random walk), lo cual es difícil de vencer en ventanas cortas.
- El hold-out de solo 6 meses (jul–dic 2025) es una muestra muy pequeña para discriminar entre modelos; incluye el periodo de transición lluviosa→seca, donde el SARIMAX con D=1 puede producir predicciones estacionales inestables por falta de ciclos completos de entrenamiento.

**Selección por RMSE → ARIMAX.** Sin embargo, como se discutió en la Fase 4, el ARIMAX tiene una **matriz de covarianza singular**, lo que invalida la inferencia sobre sus coeficientes. Esto crea un dilema:
- El modelo numéricamente estable (SARIMAX) predice peor fuera de muestra.
- El modelo que predice mejor (ARIMAX) tiene inferencia estadística no confiable.

Para fines de la **hipótesis de investigación** (efecto del parque vehicular), los coeficientes del **SARIMAX** son los únicos estadísticamente interpretables, pese a su peor RMSE. El diagnóstico de residuales del ARIMAX que sigue debe leerse con esta salvedad.


In [ ]:
# =============================================================================
# 5.3 Visualización ajuste vs. realidad en el hold-out
# =============================================================================
plt.figure(figsize=(13, 5))
plt.plot(y_train.index, y_train, color="#34495e", label="Train (real)")
plt.plot(y_test.index, y_test, color="#c0392b", marker="o", label="Test (real)")
plt.plot(y_test.index, yhat1, "--", color="#2980b9", marker="s", ms=4, label="ARIMAX pred")
plt.plot(y_test.index, yhat2, "--", color="#27ae60", marker="^", ms=4, label="SARIMAX pred")
plt.axvline(y_test.index[0], color="gray", ls=":", lw=1)
plt.title("PM2.5: ajuste fuera de muestra (hold-out)")
plt.ylabel("μg/m³"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 5.4 Diagnóstico de residuales del modelo ganador
# =============================================================================
resid = ganador.resid.dropna()

# --- Ljung-Box: H0 = no autocorrelación en residuales ---
lb = acorr_ljungbox(resid, lags=[6, 12], return_df=True)
print("--- Ljung-Box (H0: residuales NO autocorrelacionados) ---")
display(lb)

# --- Jarque-Bera: H0 = normalidad ---
jb_stat, jb_p, skew, kurt = jarque_bera(resid)
print(f"\n--- Jarque-Bera --- estad={jb_stat:.3f} | p={jb_p:.4f} | "
      f"skew={skew:.3f} | kurtosis={kurt:.3f}")
print("Residuales normales." if jb_p > 0.05 else "Se rechaza normalidad (p<0.05).")

# --- Breusch-Pagan: H0 = homocedasticidad (residuales vs. tiempo) ---
exog_bp = np.column_stack([np.ones(len(resid)), np.arange(len(resid))])
bp_stat, bp_p, _, _ = het_breuschpagan(resid, exog_bp)
print(f"\n--- Breusch-Pagan --- estad={bp_stat:.3f} | p={bp_p:.4f}")
print("Homocedástico." if bp_p > 0.05 else "Indicio de heterocedasticidad (p<0.05).")


### Interpretación del diagnóstico de residuales (ARIMAX)

| Prueba | Estadístico | p-valor | Veredicto |
|---|---|---|---|
| **Ljung-Box (lag 6)** | 1.15 | 0.979 | ✅ Pasa. No hay autocorrelación residual significativa hasta 6 meses. |
| **Ljung-Box (lag 12)** | 14.53 | 0.268 | ✅ Pasa. No hay autocorrelación estacional residual. |
| **Jarque-Bera** | 37.72 | 0.0000 | ❌ Falla. Residuales **no normales** (skewness = 1.68, kurtosis = 6.37). |
| **Breusch-Pagan** | 0.036 | 0.851 | ✅ Pasa. Residuales homocedásticos. |

**Análisis de las violaciones:**

La fuerte **asimetría positiva** (skew = 1.68) y **curtosis excesiva** (6.37, vs. 3 para una normal) en los residuales del ARIMAX indican que hay meses con picos de PM2.5 que el modelo no captura (colas pesadas a la derecha). Esto es consistente con eventos puntuales — quemas agrícolas, incendios forestales en Petén, o inversiones térmicas severas — que no están representados por las variables del modelo.

**Consecuencias prácticas:**
- Los errores estándar (y por tanto los p-valores) del ARIMAX están subestimados. Dado que el p-valor del coeficiente vehicular ya es 0.939, esto no cambia la conclusión para este modelo, pero refuerza la **no confiabilidad** de sus intervalos de confianza.
- Para el SARIMAX (que en su propio resumen mostró Jarque-Bera p=0.70, es decir residuales aproximadamente normales), la inferencia sobre los coeficientes está en mejor posición.


In [ ]:
# =============================================================================
# 5.5 Gráficos de diagnóstico estándar de statsmodels
# =============================================================================
ganador.plot_diagnostics(figsize=(13, 9))
plt.suptitle(f"Diagnóstico de residuales — {nombre_g}", y=1.01)
plt.tight_layout(); plt.show()


---
# Fase 6 · Deployment

En esta fase final no desplegamos un servicio en producción, sino que **traducimos los resultados estadísticos a conclusiones de investigación**: ¿existe evidencia de que el parque vehicular incide en el PM2.5, una vez controlado el clima?


In [ ]:
# =============================================================================
# 6.1 Extracción e interpretación de coeficientes exógenos
# =============================================================================
def tabla_coeficientes(modelo, exog_names, nombre):
    #\"\"\"Construye tabla de coeficientes exógenos con p-valor y significancia.\"\"\"
    params  = modelo.params
    pvalues = modelo.pvalues
    conf    = modelo.conf_int()
    filas = []
    for v in exog_names:
        if v in params.index:
            p = pvalues[v]
            sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "ns"
            filas.append({
                "Variable": v,
                "Coef": round(params[v], 4),
                "p-valor": round(p, 4),
                "IC95%_inf": round(conf.loc[v, 0], 4),
                "IC95%_sup": round(conf.loc[v, 1], 4),
                "Signif.": sig,
            })
    print(f"=== Coeficientes exógenos — {nombre} ===")
    return pd.DataFrame(filas).set_index("Variable")

tab_arimax  = tabla_coeficientes(arimax,  EXOG_ARIMAX,  "ARIMAX")
display(tab_arimax)
tab_sarimax = tabla_coeficientes(sarimax, EXOG_SARIMAX, "SARIMAX")
display(tab_sarimax)
print("Códigos: *** p<0.01 | ** p<0.05 | * p<0.10 | ns = no significativo")


In [ ]:
# =============================================================================
# 6.2 Conclusión automatizada sobre la hipótesis de investigación
# =============================================================================
tab_g = tab_sarimax if nombre_g == "SARIMAX" else tab_arimax
veh_row = tab_g.loc["VEH_ACUMULADO"]
coef, pval = veh_row["Coef"], veh_row["p-valor"]

print("="*72)
print("CONCLUSIÓN ESTADÍSTICA — efecto del parque vehicular sobre PM2.5")
print("="*72)
print(f"Modelo ganador      : {nombre_g}")
print(f"Coeficiente VEH     : {coef}  (sobre exógena estandarizada)")
print(f"p-valor             : {pval}")
print("-"*72)
if pval < 0.05:
    signo = "positiva" if coef > 0 else "negativa"
    print(f"Se RECHAZA H0 (p<0.05): existe una relación {signo} y estadísticamente")
    print("significativa entre el crecimiento del parque vehicular liviano y PM2.5.")
    print(f"Interpretación: a mayor parque vehicular acumulado, el PM2.5 mensual")
    print(f"{'aumenta' if coef>0 else 'disminuye'} de forma significativa, controlando por clima.")
else:
    print("NO se rechaza H0 (p>=0.05): con estos datos NO hay evidencia suficiente")
    print("de un efecto significativo del parque vehicular sobre PM2.5. Posibles causas:")
    print("- Tamaño muestral reducido (~46 meses) que limita la potencia estadística.")
    print("- Colinealidad con variables climáticas o estacionalidad que absorbe el efecto.")
    print("- El stock vehicular varía suavemente y aporta poca variación dentro del periodo.")


## 6.3 Lectura técnica de resultados con datos reales

### Resultado del ARIMAX (modelo ganador por RMSE)

El ARIMAX(1,0,3) arrojó un coeficiente de `VEH_ACUMULADO` = **+4.51** con p-valor = **0.939** y un intervalo de confianza de [−110.89, +119.92]. No se rechaza H₀. Sin embargo, este resultado es **estadísticamente vacío**: la matriz de covarianza es singular (condición = 7.72×10¹⁴) y los errores estándar son numéricamente inestables. El modelo es útil para predicción a corto plazo (RMSE = 15.73 μg/m³) pero no para inferencia sobre el efecto vehicular.

### Resultado del SARIMAX (modelo con inferencia confiable)

El SARIMAX(0,0,2)×(1,1,1,12) es el único modelo con covarianza estable y residuales aproximadamente normales (Jarque-Bera p=0.70). Sus hallazgos sobre las exógenas son interpretables:

- **`VEH_ACUMULADO`:** coef = **−19.10**, p = **0.003** (significativo al 1%), IC 95% [−31.67, −6.52]. El signo es **negativo**, contrario a H₁. Esto sugiere que, controlando por clima y estacionalidad, el crecimiento acumulado del parque vehicular se asocia con una *disminución* en PM2.5. Este resultado probablemente refleja una **confusión con la tendencia temporal** (más vehículos = más tiempo transcurrido) y/o el efecto de **renovación de flota** (los vehículos nuevos emiten menos que los antiguos que desplazamiento). No debe interpretarse como "más vehículos mejoran el aire".

- **`TEMPERATURA_MEDIA`:** coef = **+21.23**, p = **0.0003** (significativo al 0.1%), IC 95% [+9.72, +32.74]. Es el predictor más robusto del modelo. Su interpretación es meteorológica: temperaturas más altas ocurren en la época seca (nov–abr), cuando la falta de lluvia y las inversiones térmicas acumulan contaminantes. Este resultado es consistente con la literatura de calidad del aire en valles tropicales de altitud.

- **`PRECIPITACION`:** coef = −2.49, p = 0.188. Dirección esperada (la lluvia lava el PM2.5) pero no alcanza significancia estadística, probablemente porque la suma mensual suaviza el efecto puntual de los eventos de lluvia.

- **`HUMEDAD_RELATIVA`:** coef = +17.08, p = 0.119. Marginalmente no significativo. La dirección positiva podría reflejar procesos de higroscopia del PM2.5 (las partículas absorben agua y aumentan su masa medida) o colinealidad con la temperatura.

- **`VELOCIDAD_VIENTO`:** coef = +4.63, p = 0.348. No significativo. La dirección positiva es inesperada (se esperaría que más viento disperse contaminantes); esto puede indicar que en la escala mensual el efecto dispersivo se diluye o que el viento también arrastra partículas de fuentes externas.

### Limitaciones del SARIMAX observadas

Pese a que es el modelo con inferencia más confiable, presenta dos violaciones de supuestos:
- **Ljung-Box lag 1: p=0.01** → autocorrelación residual significativa, sugiere que queda estructura temporal no capturada.
- **Heterocedasticidad H=27.32, p=0.01** → la varianza de los residuales no es constante, lo que puede sesgar los errores estándar hacia abajo. Una corrección con `cov_type='robust'` (errores HAC) sería apropiada como verificación de robustez.

## 6.4 Conclusión para la tesis

**Respuesta a la hipótesis de investigación:**

Con los datos disponibles (46 meses, marzo 2022 – diciembre 2025), **no se encuentra evidencia de que el aumento del parque vehicular liviano incremente la concentración de PM2.5** en la Ciudad de Guatemala:

- En el modelo ARIMAX, el efecto es no significativo (p=0.939) y la inferencia no es confiable (matriz singular).
- En el modelo SARIMAX (inferencia estable), el efecto es estadísticamente significativo (p=0.003) pero con **signo negativo** — lo opuesto a la hipótesis.

**Este "no hallazgo" no significa que los vehículos no contaminan.** Significa que con esta ventana temporal, esta resolución (mensual), y esta metodología (regresor de stock acumulado), el efecto vehicular no es separable del efecto del tiempo, la estacionalidad climática y la renovación de flota. La variable dominante sobre PM2.5 resultó ser la **temperatura media** (proxy de la época seca/lluviosa), no el parque vehicular.

## 6.5 Recomendaciones metodológicas

Para fortalecer la investigación en versiones futuras:

1. **Usar la tasa de crecimiento (Δ% mensual) en lugar del acumulado monotónico** como variable vehicular, o los flujos (`VEH_ALTAS`) directamente, para evitar la confusión con la tendencia temporal.
2. **Estimar el parque vehicular *circulante* neto** (altas − bajas − desregistros), no solo las altas, usando los datos de SAT de baja que ya solicitaste.
3. **Considerar modelos con tendencia determinística** (intercepto + pendiente temporal) junto con el regresor vehicular para aislar el efecto del paso del tiempo.
4. **Ampliar la ventana temporal** más allá de 46 meses si las mediciones de PM2.5 anteriores a marzo 2022 se vuelven disponibles.
5. **Aplicar transformación logarítmica a PM2.5** para reducir la asimetría y los outliers que afectaron la normalidad de los residuales del ARIMAX.
6. **Reestimar el SARIMAX con errores robustos** (`cov_type='robust_approx'` o `'HAC'`) para verificar si la significancia del coeficiente vehicular sobrevive la corrección por heterocedasticidad.
7. **Reportar ambos modelos** en la tesis, presentando el ARIMAX como modelo predictivo y el SARIMAX como modelo inferencial, explicando la paradoja AIC vs. RMSE.

## 6.6 Persistencia del modelo (opcional)
```python
ganador.save('modelo_pm25_ganador.pkl')
# Recarga:  from statsmodels.tsa.statespace.sarimax import SARIMAXResults
#           modelo = SARIMAXResults.load('modelo_pm25_ganador.pkl')
```
